# 1. Установка библиотеки для Selenium

In [25]:
# %pip install selenium webdriver-manager beautifulsoup4

# 2. Парсинг ВБ

Так как сайт WB защищен от копирования данных, а версия API предоставлена только для продавцов, для доступа к которой необходим аккаунт продавца, ссылку на сайт не удалось использовать в коде. Было принято решение попробовать вручную копировать данные сайта. В качестве данных служит файл - wb_smeshariki.js.

Однако постоянное копирование даннных с сайта в локальный файл - та еще морока. Для удобства этот процесс был автоматизирован с помощью DevTools.

Чтобы каждый раз вручную не копировать JSON код страницы, будем использовать Playwright. Python сам открывает настоящий браузер, заходит на WB, ловит нужный JSON-ответ из Network и сохраняет его. Это уже не костыль, а нормальная автоматизация DevTools.

Selenium открывает WB → DevTools-логи ловят JSON → Python собирает DataFrame → сохраняет CSV.

Установка playwright

In [26]:

# %pip install playwright pandas

In [27]:
# %pip install playwright

In [28]:
# !playwright install chromium

### Парсинг данных

На маркетплейсе Wildberries по запросу **«Смешарики»** на момент сбора данных было представлено более **139 тысяч товаров**. Однако значительная часть продукции может содержать упоминание бренда или персонажей без официальной лицензии. Поэтому для повышения качества выборки в исследование были включены только товары, отмеченные меткой **«Оригинал»**, а также товары из официальных магазинов **«Смешарики»** и **Riki Shop**.

Сбор данных осуществлялся двумя способами:

1. **Парсинг официальных магазинов «Смешарики» и Riki Shop.**

   Для каждого магазина выполнялась автоматическая прокрутка страницы с последующим извлечением информации о товарах через сетевые запросы браузера (DevTools). Такой подход позволил получить полный ассортимент продукции, представленной в официальных магазинах бренда.

2. **Парсинг поисковой выдачи Wildberries по запросу «Смешарики» с фильтром «Оригинал».**

   В ходе анализа сетевого трафика страницы был обнаружен внутренний запрос Wildberries, содержащий данные о товарах поисковой выдачи. Сбор данных осуществлялся постранично путем обращения к данному запросу и извлечения информации о товарах из JSON-ответов. Это позволило получить полный список лицензионной продукции, представленной сторонними производителями. В выборку вошли товары таких брендов, как **SELA**, **Brick Labs**, **Hatber**, **Эксмо**, **Издательство АСТ**, **Мульти-пульти** и других производителей, выпускающих продукцию по лицензии бренда «Смешарики».

После объединения данных из всех источников были удалены дубликаты товаров и выполнена дополнительная фильтрация по ключевым словам, связанным с брендом и персонажами «Смешариков». В результате был сформирован единый датасет, содержащий как товары официальных магазинов, так и лицензионную продукцию партнеров бренда.

Общая работа для всех ссылок

In [ ]:
# import json
# import time
# import base64
# import pandas as pd
# from urllib.parse import urlencode

# from selenium import webdriver
# from selenium.webdriver.chrome.service import Service
# from webdriver_manager.chrome import ChromeDriverManager


# brand_pages = {
#     'Смешарики': 'https://www.wildberries.ru/brands/smeshariki',
#     'Riki Shop': 'https://www.wildberries.ru/brands/311585829-riki-shop'
# }

# search_api_url = (
#     'https://www.wildberries.ru/__internal/u-search/exactmatch/ru/common/v18/search'
# )

# search_params = {
#     'ab_testid': 'catboost_exp3_1',
#     'appType': 1,
#     'curr': 'rub',
#     'dest': -1257786,
#     'foriginal': 1,
#     'hide_dtype': 15,
#     'hide_vflags': 4294967296,
#     'inheritFilters': 'false',
#     'lang': 'ru',
#     'locale': 'ru',
#     'query': 'смешарики',
#     'resultset': 'catalog',
#     'sort': 'popular',
#     'spp': 30,
#     'suppressSpellcheck': 'false'
# }

# keywords = [
#     'смешарик', 'смешарики', 'крош', 'нюша', 'бараш',
#     'ёжик', 'ежик', 'лосяш', 'кар-карыч', 'копатыч',
#     'совунья', 'пин'
# ]


# def get_products_from_json(data):
#     products = []

#     if isinstance(data, dict):
#         if isinstance(data.get('products'), list):
#             products.extend(data.get('products'))

#         if isinstance(data.get('data'), dict):
#             data_products = data.get('data', {}).get('products', [])

#             if isinstance(data_products, list):
#                 products.extend(data_products)

#     return products


# def get_response_body(driver, request_id):
#     body = driver.execute_cdp_cmd(
#         'Network.getResponseBody',
#         {'requestId': request_id}
#     )

#     response_body = body.get('body', '')

#     if body.get('base64Encoded'):
#         response_body = base64.b64decode(response_body).decode('utf-8')

#     return response_body


# def collect_brand_page(driver, page_url, source_name):
#     driver.get_log('performance')
#     driver.get(page_url)

#     print()
#     print('Открыли магазин:', source_name)

#     time.sleep(15)

#     products_data = []
#     seen_ids = set()

#     last_count = 0
#     no_new_scrolls = 0

#     for i in range(120):
#         driver.execute_script(
#             'window.scrollTo(0, document.body.scrollHeight);'
#         )
#         time.sleep(2)

#         logs = driver.get_log('performance')

#         for log in logs:
#             try:
#                 message = json.loads(log['message'])['message']
#             except Exception:
#                 continue

#             if message.get('method') != 'Network.responseReceived':
#                 continue

#             response = message.get('params', {}).get('response', {})
#             request_id = message.get('params', {}).get('requestId')
#             url = response.get('url', '')

#             if 'u-catalog/brands' not in url:
#                 continue

#             try:
#                 data = json.loads(get_response_body(driver, request_id))
#                 products = get_products_from_json(data)

#                 for item in products:
#                     product_id = item.get('id')

#                     if product_id not in seen_ids:
#                         seen_ids.add(product_id)
#                         item['source_name'] = source_name
#                         item['source_type'] = 'brand_page'
#                         item['source_page'] = None
#                         products_data.append(item)

#             except Exception:
#                 pass

#         current_count = len(seen_ids)

#         print(f'Скролл {i + 1}: товаров собрано {current_count}')

#         if current_count == last_count:
#             no_new_scrolls += 1
#         else:
#             no_new_scrolls = 0

#         last_count = current_count

#         if no_new_scrolls >= 4:
#             print('Новые товары больше не подгружаются.')
#             break

#     print(f'Итого собрано для "{source_name}": {len(products_data)}')
#     return products_data


# def collect_original_search(driver, max_pages=30):
#     print()
#     print('Собираем поиск: Смешарики + Оригинал')

#     start_page = (
#         'https://www.wildberries.ru/catalog/0/search.aspx'
#         '?page=1&sort=popular&search=смешарики&foriginal=1&meta_charcs=false'
#     )

#     driver.get(start_page)
#     time.sleep(10)

#     products_data = []
#     seen_ids = set()
#     empty_pages = 0

#     for page in range(1, max_pages + 1):
#         params = search_params.copy()
#         params['page'] = page

#         api_url = search_api_url + '?' + urlencode(params)

#         script = """
#             const callback = arguments[arguments.length - 1];
#             fetch(arguments[0])
#                 .then(response => response.json())
#                 .then(data => callback(data))
#                 .catch(error => callback({'error': String(error)}));
#         """

#         data = driver.execute_async_script(script, api_url)

#         products = get_products_from_json(data)

#         new_count = 0

#         for item in products:
#             product_id = item.get('id')

#             if product_id not in seen_ids:
#                 seen_ids.add(product_id)
#                 item['source_name'] = 'Поиск Смешарики + Оригинал'
#                 item['source_type'] = 'original_search'
#                 item['source_page'] = page
#                 products_data.append(item)
#                 new_count += 1

#         print(
#             f'page={page}: товаров в ответе {len(products)}, '
#             f'новых {new_count}'
#         )

#         if len(products) == 0:
#             empty_pages += 1
#         else:
#             empty_pages = 0

#         if empty_pages >= 2:
#             print('Две пустые страницы подряд. Останавливаем поиск.')
#             break

#         time.sleep(1)

#     print(f'Итого собрано из поиска Оригинал: {len(products_data)}')
#     return products_data


# options = webdriver.ChromeOptions()
# options.add_argument('--start-maximized')
# options.add_argument('--disable-blink-features=AutomationControlled')

# options.add_experimental_option(
#     'excludeSwitches',
#     ['enable-automation']
# )

# options.add_experimental_option(
#     'useAutomationExtension',
#     False
# )

# options.set_capability(
#     'goog:loggingPrefs',
#     {'performance': 'ALL'}
# )

# driver = webdriver.Chrome(
#     service=Service(ChromeDriverManager().install()),
#     options=options
# )

# driver.execute_script("""
# Object.defineProperty(
#     navigator,
#     'webdriver',
#     {
#         get: () => undefined
#     }
# )
# """)


# all_products = []

# for brand_name, brand_url in brand_pages.items():
#     products = collect_brand_page(
#         driver=driver,
#         page_url=brand_url,
#         source_name=brand_name
#     )

#     all_products.extend(products)


# products_original = collect_original_search(
#     driver=driver,
#     max_pages=30
# )

# all_products.extend(products_original)

# driver.quit()


# rows = []

# for item in all_products:
#     product_id = item.get('id')
#     sizes = item.get('sizes', [])

#     price = None
#     discount_price = None

#     if len(sizes) > 0:
#         price_info = sizes[0].get('price', {})
#         price = price_info.get('basic')
#         discount_price = price_info.get('product')

#     rows.append({
#         'product_id': product_id,
#         'product_name': item.get('name'),
#         'brand': item.get('brand'),
#         'brand_id': item.get('brandId'),
#         'source_name': item.get('source_name'),
#         'source_type': item.get('source_type'),
#         'source_page': item.get('source_page'),
#         'seller': item.get('supplier'),
#         'seller_id': item.get('supplierId'),
#         'seller_rating': item.get('supplierRating'),
#         'price': price / 100 if price else None,
#         'discount_price': discount_price / 100 if discount_price else None,
#         'rating': item.get('reviewRating'),
#         'reviews_count': item.get('feedbacks'),
#         'category_name': item.get('entity'),
#         'subject_id': item.get('subjectId'),
#         'subject_parent_id': item.get('subjectParentId'),
#         'total_quantity': item.get('totalQuantity'),
#         'marketplace': 'Wildberries',
#         'link': f'https://www.wildberries.ru/catalog/{product_id}/detail.aspx'
#     })

# df_wb = pd.DataFrame(rows)

# if len(df_wb) > 0:
#     df_wb = df_wb.drop_duplicates(subset=['product_id'])

#     text_cols = ['product_name', 'brand', 'seller', 'category_name']

#     for col in text_cols:
#         df_wb[col] = df_wb[col].fillna('')

#     df_wb['text_for_filter'] = (
#         df_wb['product_name'] + ' ' +
#         df_wb['brand'] + ' ' +
#         df_wb['seller'] + ' ' +
#         df_wb['category_name']
#     ).str.lower()

#     mask_brand_page = df_wb['source_type'] == 'brand_page'

#     mask_keywords = df_wb['text_for_filter'].apply(
#         lambda x: any(word in x for word in keywords)
#     )

#     df_wb = df_wb[
#         mask_brand_page | mask_keywords
#     ].copy()

#     df_wb = df_wb.drop(columns=['text_for_filter'])

# display(df_wb.head())
# print(df_wb.shape)

# df_wb.to_csv(
#     'smeshariki_wb_final.csv',
#     index=False,
#     encoding='utf-8-sig'
# )

# print('Файл сохранён: smeshariki_wb_final.csv')

# print()
# print('Распределение по источникам:')
# display(df_wb['source_name'].value_counts())

# print()
# print('Страницы поиска Оригинал:')
# display(
#     df_wb[df_wb['source_type'] == 'original_search']
#     ['source_page']
#     .value_counts()
#     .sort_index()
# )

# print()
# print('Топ брендов:')
# display(df_wb['brand'].value_counts().head(30))

# print()
# print('Топ продавцов:')
# display(df_wb['seller'].value_counts().head(30))

Сохранение в эксель на всякий случай

## Блок 1. Импорт библиотек и настройки

Подключаются библиотеки для парсинга, обработки данных и работы с браузером.

Задаются ссылки на официальные магазины "Смешарики" и "Riki shop".

Настраиваются параметры поискового запроса «Смешарики + Оригинал».

Формируется список ключевых слов для последующей фильтрации товаров.

In [30]:
import json
import time
import base64
import pandas as pd
from urllib.parse import urlencode

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager


brand_pages = {
    'Смешарики': 'https://www.wildberries.ru/brands/smeshariki',
    'Riki Shop': 'https://www.wildberries.ru/brands/311585829-riki-shop'
}

search_api_url = (
    'https://www.wildberries.ru/__internal/u-search/exactmatch/ru/common/v18/search'
)

search_params = {
    'ab_testid': 'catboost_exp3_1',
    'appType': 1,
    'curr': 'rub',
    'dest': -1257786,
    'foriginal': 1,
    'hide_dtype': 15,
    'hide_vflags': 4294967296,
    'inheritFilters': 'false',
    'lang': 'ru',
    'locale': 'ru',
    'query': 'смешарики',
    'resultset': 'catalog',
    'sort': 'popular',
    'spp': 30,
    'suppressSpellcheck': 'false'
}

keywords = [
    'смешарик', 'смешарики', 'крош', 'нюша', 'бараш',
    'ёжик', 'ежик', 'лосяш', 'кар-карыч', 'копатыч',
    'совунья', 'пин'
]

## Блок 2. Функции парсинга
get_products_from_json() извлекает список товаров из JSON-ответа Wildberries.

get_response_body() получает содержимое сетевого запроса через DevTools.

collect_brand_page() собирает товары со страницы бренда с помощью автоматической прокрутки.

collect_original_search() собирает товары из оригинальной поисковой выдачи по страницам через внутренний API Wildberries.

In [31]:
def get_products_from_json(data):
    products = []

    if isinstance(data, dict):
        if isinstance(data.get('products'), list):
            products.extend(data.get('products'))

        if isinstance(data.get('data'), dict):
            data_products = data.get('data', {}).get('products', [])

            if isinstance(data_products, list):
                products.extend(data_products)

    return products


def get_response_body(driver, request_id):
    body = driver.execute_cdp_cmd(
        'Network.getResponseBody',
        {'requestId': request_id}
    )

    response_body = body.get('body', '')

    if body.get('base64Encoded'):
        response_body = base64.b64decode(response_body).decode('utf-8')

    return response_body


def collect_brand_page(driver, page_url, source_name):
    driver.get_log('performance')
    driver.get(page_url)

    print()
    print('Открыли магазин:', source_name)

    time.sleep(15)

    products_data = []
    seen_ids = set()

    last_count = 0
    no_new_scrolls = 0

    for i in range(120):
        driver.execute_script(
            'window.scrollTo(0, document.body.scrollHeight);'
        )
        time.sleep(2)

        logs = driver.get_log('performance')

        for log in logs:
            try:
                message = json.loads(log['message'])['message']
            except Exception:
                continue

            if message.get('method') != 'Network.responseReceived':
                continue

            response = message.get('params', {}).get('response', {})
            request_id = message.get('params', {}).get('requestId')
            url = response.get('url', '')

            if 'u-catalog/brands' not in url:
                continue

            try:
                data = json.loads(get_response_body(driver, request_id))
                products = get_products_from_json(data)

                for item in products:
                    product_id = item.get('id')

                    if product_id not in seen_ids:
                        seen_ids.add(product_id)
                        item['source_name'] = source_name
                        item['source_type'] = 'brand_page'
                        item['source_page'] = None
                        products_data.append(item)

            except Exception:
                pass

        current_count = len(seen_ids)

        print(f'Скролл {i + 1}: товаров собрано {current_count}')

        if current_count == last_count:
            no_new_scrolls += 1
        else:
            no_new_scrolls = 0

        last_count = current_count

        if no_new_scrolls >= 4:
            print('Новые товары больше не подгружаются.')
            break

    print(f'Итого собрано для "{source_name}": {len(products_data)}')
    return products_data


def collect_original_search(driver, max_pages=30):
    print()
    print('Собираем поиск: Смешарики + Оригинал')

    start_page = (
        'https://www.wildberries.ru/catalog/0/search.aspx'
        '?page=1&sort=popular&search=смешарики&foriginal=1&meta_charcs=false'
    )

    driver.get(start_page)
    time.sleep(10)

    products_data = []
    seen_ids = set()
    empty_pages = 0

    for page in range(1, max_pages + 1):
        params = search_params.copy()
        params['page'] = page

        api_url = search_api_url + '?' + urlencode(params)

        script = """
            const callback = arguments[arguments.length - 1];
            fetch(arguments[0])
                .then(response => response.json())
                .then(data => callback(data))
                .catch(error => callback({'error': String(error)}));
        """

        data = driver.execute_async_script(script, api_url)

        products = get_products_from_json(data)

        new_count = 0

        for item in products:
            product_id = item.get('id')

            if product_id not in seen_ids:
                seen_ids.add(product_id)
                item['source_name'] = 'Поиск Смешарики + Оригинал'
                item['source_type'] = 'original_search'
                item['source_page'] = page
                products_data.append(item)
                new_count += 1

        print(
            f'page={page}: товаров в ответе {len(products)}, '
            f'новых {new_count}'
        )

        if len(products) == 0:
            empty_pages += 1
        else:
            empty_pages = 0

        if empty_pages >= 2:
            print('Две пустые страницы подряд. Останавливаем поиск.')
            break

        time.sleep(1)

    print(f'Итого собрано из поиска Оригинал: {len(products_data)}')
    return products_data

## Блок 3. Запуск браузера
Настраивается ChromeDriver.

Включается сбор сетевых запросов DevTools.

Выполняется маскировка Selenium для снижения вероятности блокировки со стороны сайта.

In [32]:
options = webdriver.ChromeOptions()
options.add_argument('--start-maximized')
options.add_argument('--disable-blink-features=AutomationControlled')

options.add_experimental_option(
    'excludeSwitches',
    ['enable-automation']
)

options.add_experimental_option(
    'useAutomationExtension',
    False
)

options.set_capability(
    'goog:loggingPrefs',
    {'performance': 'ALL'}
)

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

driver.execute_script("""
Object.defineProperty(
    navigator,
    'webdriver',
    {
        get: () => undefined
    }
)
""")

## Блок 4. Сбор данных
Собираются товары из официального магазина «Смешарики», Riki Shop.

Собираются все товары из оригинальной поисковой выдачи по запросу «Смешарики».

Все данные объединяются в единый список.

In [33]:
all_products = []

for brand_name, brand_url in brand_pages.items():
    products = collect_brand_page(
        driver=driver,
        page_url=brand_url,
        source_name=brand_name
    )

    all_products.extend(products)


products_original = collect_original_search(
    driver=driver,
    max_pages=30
)

all_products.extend(products_original)

driver.quit()


Открыли магазин: Смешарики
Скролл 1: товаров собрано 100
Скролл 2: товаров собрано 200
Скролл 3: товаров собрано 200
Скролл 4: товаров собрано 300
Скролл 5: товаров собрано 300
Скролл 6: товаров собрано 400
Скролл 7: товаров собрано 400
Скролл 8: товаров собрано 500
Скролл 9: товаров собрано 500
Скролл 10: товаров собрано 600
Скролл 11: товаров собрано 600
Скролл 12: товаров собрано 700
Скролл 13: товаров собрано 700
Скролл 14: товаров собрано 800
Скролл 15: товаров собрано 800
Скролл 16: товаров собрано 900
Скролл 17: товаров собрано 900
Скролл 18: товаров собрано 1000
Скролл 19: товаров собрано 1000
Скролл 20: товаров собрано 1100
Скролл 21: товаров собрано 1100
Скролл 22: товаров собрано 1200
Скролл 23: товаров собрано 1200
Скролл 24: товаров собрано 1300
Скролл 25: товаров собрано 1300
Скролл 26: товаров собрано 1400
Скролл 27: товаров собрано 1400
Скролл 28: товаров собрано 1500
Скролл 29: товаров собрано 1500
Скролл 30: товаров собрано 1600
Скролл 31: товаров собрано 1600
Скролл

## Блок 5. Формирование датафрейма
Из полученных JSON-данных извлекаются необходимые характеристики товаров.

Формируется таблица Pandas DataFrame.

Для каждого товара сохраняются название, бренд, продавец, цена, рейтинг, количество отзывов, категория и ссылка.

In [34]:
rows = []

for item in all_products:
    product_id = item.get('id')
    sizes = item.get('sizes', [])

    price = None
    discount_price = None

    if len(sizes) > 0:
        price_info = sizes[0].get('price', {})
        price = price_info.get('basic')
        discount_price = price_info.get('product')

    rows.append({
        'product_id': product_id,
        'product_name': item.get('name'),
        'brand': item.get('brand'),
        'brand_id': item.get('brandId'),
        'source_name': item.get('source_name'),
        'source_type': item.get('source_type'),
        'source_page': item.get('source_page'),
        'seller': item.get('supplier'),
        'seller_id': item.get('supplierId'),
        'seller_rating': item.get('supplierRating'),
        'price': price / 100 if price else None,
        'discount_price': discount_price / 100 if discount_price else None,
        'rating': item.get('reviewRating'),
        'reviews_count': item.get('feedbacks'),
        'category_name': item.get('entity'),
        'subject_id': item.get('subjectId'),
        'subject_parent_id': item.get('subjectParentId'),
        'total_quantity': item.get('totalQuantity'),
        'marketplace': 'Wildberries',
        'link': f'https://www.wildberries.ru/catalog/{product_id}/detail.aspx'
    })

df_wb = pd.DataFrame(rows)

## Блок 6. Очистка и фильтрация данных
Удаляются дубликаты товаров.

Заполняются пропущенные текстовые значения.

Выполняется фильтрация по ключевым словам, связанным со Смешариками.

Сохраняются товары из официальных магазинов и релевантные товары из поиска.

In [35]:
if len(df_wb) > 0:
    df_wb = df_wb.drop_duplicates(subset=['product_id'])

    text_cols = ['product_name', 'brand', 'seller', 'category_name']

    for col in text_cols:
        df_wb[col] = df_wb[col].fillna('')

    df_wb['text_for_filter'] = (
        df_wb['product_name'] + ' ' +
        df_wb['brand'] + ' ' +
        df_wb['seller'] + ' ' +
        df_wb['category_name']
    ).str.lower()

    mask_brand_page = df_wb['source_type'] == 'brand_page'

    mask_keywords = df_wb['text_for_filter'].apply(
        lambda x: any(word in x for word in keywords)
    )

    df_wb = df_wb[
        mask_brand_page | mask_keywords
    ].copy()

    df_wb = df_wb.drop(columns=['text_for_filter'])

display(df_wb.head())
print(df_wb.shape)

,product_id,product_name,brand,brand_id,source_name,source_type,source_page,seller,seller_id,seller_rating,price,discount_price,rating,reviews_count,category_name,subject_id,subject_parent_id,total_quantity,marketplace,link
0,183944199,Мультивселенная. Комиксы BUBBLE,Смешарики,1187,Смешарики,brand_page,NaN,BUBBLE,38137,4.9,2608.0,990.0,4.9,792,,4961,786,40,Wildberries,https://www.wildberries.ru/catalog/183944199/d...
1,158386267,Школьные мелки 10 цветов 29 штук,Смешарики,1187,Смешарики,brand_page,NaN,ТойсМаркет,339332,4.9,535.0,168.0,4.3,241,,714,571,40,Wildberries,https://www.wildberries.ru/catalog/158386267/d...
2,217050192,Маленькая мягкая игрушка брелок Крош на рюкзак,Смешарики,1187,Смешарики,brand_page,NaN,Фабрика игрушек МЯКИШИ.,27411,4.9,1145.0,385.0,4.9,1155,игрушки-подвески,268,7,40,Wildberries,https://www.wildberries.ru/catalog/217050192/d...
3,256270492,Термостакан 350 мл с клапаном,Смешарики,1187,Смешарики,brand_page,NaN,Бытпласт - товары для дома из пластика,80145,4.9,1200.0,331.0,4.9,350,,1274,1590,40,Wildberries,https://www.wildberries.ru/catalog/256270492/d...
4,217080342,Маленькая мягкая игрушка Пин на рюкзак,Смешарики,1187,Смешарики,brand_page,NaN,Фабрика игрушек МЯКИШИ.,27411,4.9,1436.0,398.0,4.9,696,мягкие игрушки,268,7,40,Wildberries,https://www.wildberries.ru/catalog/217080342/d...


(2835, 20)


## Блок 7. Сохранение и первичный анализ
Итоговый датасет сохраняется в форматах CSV и Excel.

Выводится размер итоговой выборки.

Анализируется распределение товаров по источникам.

Определяются наиболее представленные бренды и продавцы.

In [36]:
df_wb.to_csv(
    'smeshariki_wb_final.csv',
    index=False,
    encoding='utf-8-sig'
)

df_wb.to_excel(
    'smeshariki_wb_final.xlsx',
    index=False
)

print('Файлы сохранены: smeshariki_wb_final.csv и smeshariki_wb_final.xlsx')

print()
print('Распределение по источникам:')
display(df_wb['source_name'].value_counts())

print()
print('Страницы поиска Оригинал:')
display(
    df_wb[df_wb['source_type'] == 'original_search']
    ['source_page']
    .value_counts()
    .sort_index()
)

print()
print('Топ брендов:')
display(df_wb['brand'].value_counts().head(30))

print()
print('Топ продавцов:')
display(df_wb['seller'].value_counts().head(30))

Файлы сохранены: smeshariki_wb_final.csv и smeshariki_wb_final.xlsx

Распределение по источникам:


source_name
Смешарики                     2110
Поиск Смешарики + Оригинал     696
Riki Shop                       29
Name: count, dtype: int64


Страницы поиска Оригинал:


source_page
1.0     96
2.0     97
3.0    100
4.0     97
5.0     98
6.0     96
7.0     95
8.0     17
Name: count, dtype: int64


Топ брендов:


brand
Смешарики              2110
PrinTort                438
SELA                     40
Riki Shop                29
Brick labs               29
Мульти-пульти            28
Эксмо                    27
Дон Баллон               21
Василек                  19
Hatber                   15
Издательство АСТ         14
STEP puzzle company      11
INFANT                    9
Kinder                    8
SCANDIC                   6
Befree                    5
Uniqcute                  4
Zeriazor                  4
Стиль Жизни               3
Warmies                   3
Конфитрейд                2
512                       2
Sweet Club                2
PlayToday                 2
NNNB                      2
Сималенд                  1
Nika                      1
Name: count, dtype: int64


Топ продавцов:


seller
PrinTort                                 438
Гудс & Мо                                 69
Гудс & Mo                                 69
Официальный Магазин СИМ СИМ откройся      66
ТойсМаркет                                55
Филиппи КО                                51
Трифанов                                  47
MJShop                                    44
ИП Богданов М.Ю.                          43
Cozy Casa                                 43
SmiloDeals                                41
ЗинЛенд                                   40
Магазин                                   40
Лирика                                    39
Лемания                                   39
MELON FASHION GROUP                       39
ПРЯМОЙ КОНТРАКТ - Официальный магазин     38
ZAPADNY                                   38
Hamster                                   36
Магазин у дома                            35
Sultan Store                              34
Лакталина                                 34
ZOA

# 3. Парсинг Ozon

В отличие от Wildberries, маркетплейс Ozon предоставляет данные товаров через внутренние JSON-запросы, которые используются самим сайтом для формирования поисковой выдачи. Официальный API Ozon предназначен преимущественно для продавцов и требует авторизации, поэтому для получения данных был использован анализ сетевых запросов браузера.

Во время исследования через DevTools был найден внутренний запрос Ozon, который возвращает карточки товаров в формате JSON. Это позволило автоматизировать процесс получения данных без ручного копирования информации со страниц сайта.

Для автоматизации использовался Selenium. Браузер открывает страницу поиска Ozon, после чего Python последовательно обращается к внутреннему JSON-запросу, получает данные о товарах, формирует таблицу и сохраняет результат в файл.

Схема работы выглядит следующим образом:

Selenium открывает Ozon → Python обращается к внутреннему JSON API → извлекаются данные о товарах → выполняется очистка и фильтрация → формируется DataFrame → сохраняется CSV/XLSX.

### Парсинг данных

На маркетплейсе Ozon по запросу **«Смешарики»** представлено большое количество товаров различных производителей. Однако часть товаров может попадать в выдачу по косвенным совпадениям или рекомендациям маркетплейса. Кроме того, информация о бренде в карточках товаров отображается не всегда корректно.

Для повышения качества выборки использовались только товары, отмеченные фильтром **«Оригинал»**.

Сбор данных осуществлялся через внутренний JSON-запрос Ozon, который используется сайтом для формирования поисковой выдачи. Запрос выполнялся последовательно для нескольких страниц поиска, что позволило получить полный список товаров, удовлетворяющих условиям поиска.

После получения данных была выполнена дополнительная фильтрация по ключевым словам, связанным со вселенной «Смешарики». В выборку попадали товары, содержащие упоминания бренда или персонажей в названии товара либо бренда.

Дополнительно для каждого товара открывалась его карточка на сайте Ozon. Это было необходимо для получения информации о магазине-продавце, так как данные о продавце отсутствуют в основном JSON-ответе поисковой выдачи.

Для части товаров информация о бренде отсутствовала либо содержала некорректные значения. В таких случаях бренд восстанавливался по названию официального магазина-продавца. Например, товары магазина «Торговый Дом Эксмо» автоматически относились к бренду «Эксмо», товары магазина «МЯКИШИ» — к бренду «Мякиши», товары магазина «ХАТБЕР» — к бренду «Hatber» и т.д.

После объединения всех данных были удалены дубликаты товаров и сформирован единый датасет лицензионной продукции бренда «Смешарики», представленной на маркетплейсе Ozon.

## Блок 1. Импорт библиотек и настройки

Подключаются библиотеки для работы с браузером, обработки JSON-ответов и формирования итоговой таблицы.

Также формируется список ключевых слов, связанных со вселенной «Смешарики». В дальнейшем он используется для фильтрации товаров и удаления нерелевантных результатов поиска.

In [60]:
import json
import time
import re
import pandas as pd
from urllib.parse import urlencode

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager


keywords = [
    'смешарик', 'смешарики', 'крош', 'нюша', 'бараш',
    'ёжик', 'ежик', 'лосяш', 'кар-карыч', 'карыч',
    'копатыч', 'совунья', 'пин'
]

## Блок 2. Извлечение характеристик товара

Создаются вспомогательные функции для обработки JSON-ответов Ozon.

Функции позволяют извлекать:

- название товара;
- бренд;
- цену и цену со скидкой;
- рейтинг товара;
- количество отзывов;
- остатки товара на складе.

Также выполняется очистка текстовых и числовых значений от лишних символов и приведение данных к удобному для анализа виду.

In [61]:
def clean_price(value):
    if value is None:
        return None

    value = str(value)
    value = value.replace('₽', '')
    value = value.replace('\u2009', '')
    value = value.replace('\xa0', '')
    value = value.replace(' ', '')
    value = value.replace(',', '.')

    try:
        return float(value)
    except Exception:
        return None


def extract_name(main_state):
    for block in main_state:
        if block.get('id') == 'name':
            return block.get('textDS', {}).get('text')

    for block in main_state:
        if block.get('type') == 'textDS':
            text = block.get('textDS', {}).get('text')

            if text:
                return text

    return None


def extract_prices(main_state):
    current_price = None
    original_price = None

    for block in main_state:
        if block.get('type') != 'priceV2':
            continue

        prices = block.get('priceV2', {}).get('price', [])

        for price_item in prices:
            text = price_item.get('text')
            style = price_item.get('textStyle')

            if style == 'PRICE':
                current_price = clean_price(text)

            if style == 'ORIGINAL_PRICE':
                original_price = clean_price(text)

    return original_price, current_price


def extract_rating_reviews(main_state):
    rating = None
    reviews_count = None

    for block in main_state:
        if block.get('type') != 'labelListV2':
            continue

        items = block.get('labelListV2', {}).get('items', [])

        for item in items:
            text = item.get('text', {}).get('text')

            if not text:
                continue

            text_clean = (
                text.replace('\u2009', '')
                .replace('\xa0', ' ')
                .replace(',', '.')
            )

            if re.fullmatch(r'\d+(\.\d+)?', text_clean):
                rating = float(text_clean)

            if 'отзыв' in text_clean:
                reviews_count = re.sub(r'\D', '', text_clean)

                if reviews_count != '':
                    reviews_count = int(reviews_count)

    return rating, reviews_count


def is_bad_brand(text):
    if text is None:
        return True

    text = str(text).strip()

    if text == '':
        return True

    if text == 'Оригинал':
        return True

    if re.fullmatch(r'\d+([.,]\d+)?', text):
        return True

    if 'отзыв' in text.lower():
        return True

    return False


def extract_brand(main_state):
    for block in main_state:
        if block.get('type') != 'labelListV2':
            continue

        items = block.get('labelListV2', {}).get('items', [])

        texts = []

        for item in items:
            text = item.get('text', {}).get('text')

            if text:
                texts.append(text.strip())

        if 'Оригинал' in texts:
            for text in texts:
                if not is_bad_brand(text):
                    return text

    return None


def extract_total_quantity(main_state, item):
    for block in main_state:
        if block.get('type') == 'textDS':
            text = block.get('textDS', {}).get('text', '')

            if 'шт осталось' in text:
                quantity = re.sub(r'\D', '', text)

                if quantity != '':
                    return int(quantity)

    try:
        return (
            item.get('multiButton', {})
            .get('ozonButton', {})
            .get('addToCart', {})
            .get('quantityButton', {})
            .get('maxItems')
        )
    except Exception:
        return None

## Блок 3. Фильтрация релевантных товаров

После получения JSON-ответа маркетплейса из него извлекаются карточки товаров.

Поскольку Ozon может включать в результаты поиска рекомендованные товары и товары, не относящиеся напрямую к бренду «Смешарики», выполняется дополнительная фильтрация по ключевым словам.

В итоговый набор данных попадают только те товары, в названии или бренде которых содержатся упоминания бренда «Смешарики» либо персонажей мультсериала. Это позволяет повысить качество выборки и исключить посторонние товары из анализа.

In [62]:
def get_items_from_ozon_json(data):
    all_items = []

    widget_states = data.get('widgetStates', {})

    for key, value in widget_states.items():
        if 'tileGridDesktop' not in key:
            continue

        try:
            widget_data = json.loads(value)
        except Exception:
            continue

        items = widget_data.get('items', [])

        if isinstance(items, list):
            all_items.extend(items)

    return all_items


def is_relevant_product(product_name, brand):
    text = (
        str(product_name or '') + ' ' +
        str(brand or '')
    ).lower()

    return any(word in text for word in keywords)

## Блок 4. Получение информации о продавцах

В JSON-ответе поисковой выдачи Ozon отсутствует полноценная информация о продавце товара. Поэтому для каждого найденного товара дополнительно открывается его карточка.

После открытия страницы анализируется текстовое содержимое карточки, из которого извлекается название магазина-продавца. При этом автоматически исключаются служебные элементы интерфейса сайта, такие как кнопки, ссылки и навигационные блоки.

Полученная информация используется для дополнительного анализа структуры рынка и восстановления отсутствующих данных о брендах.

In [63]:
def is_bad_seller(text):
    if text is None:
        return True

    text = str(text).strip()

    if text == '':
        return True

    bad_values = [
        'Подписаться',
        'Заказы',
        'О магазине',
        'Перейти',
        'Чат',
        'Магазин',
        'Отзывы',
        'Вопросы о товаре',
        'Описание',
        'Характеристики'
    ]

    if text in bad_values:
        return True

    if re.fullmatch(r'\d+([.,]\d+)?', text):
        return True

    if re.fullmatch(r'\d+([.,]\d+)?\s?[KКMМ]', text):
        return True

    if 'заказ' in text.lower():
        return True

    if 'отзыв' in text.lower():
        return True

    return False


def get_seller_from_product_page(driver, product_url):
    try:
        driver.get(product_url)
        time.sleep(5)

        page_text = driver.find_element('tag name', 'body').text

        lines = [
            line.strip()
            for line in page_text.split('\n')
            if line.strip()
        ]

        for i, line in enumerate(lines):
            if line == 'Магазин':
                candidates = lines[i + 1:i + 20]

                for candidate in candidates:
                    if not is_bad_seller(candidate):
                        return candidate

    except Exception:
        return None

    return None

## Блок 5. Сбор товаров из поисковой выдачи

На данном этапе реализована основная функция парсинга данных.

Функция последовательно обращается к внутреннему JSON-запросу Ozon, который используется самим маркетплейсом для формирования поисковой выдачи. Для каждой страницы поиска выполняется получение данных о товарах, после чего информация извлекается и сохраняется во временную структуру данных.

Одновременно выполняется проверка на наличие дубликатов товаров по идентификатору товара. Это позволяет исключить повторное сохранение одинаковых карточек при переходе между страницами поиска.

Сбор продолжается до тех пор, пока маркетплейс возвращает новые товары.

In [64]:
def collect_ozon_original_search(driver, max_pages=20):
    all_products = []
    seen_ids = set()
    empty_pages = 0

    for page in range(1, max_pages + 1):
        search_url = (
            '/search/?brandcertified=t'
            '&from_global=true'
            '&opened=seller%2Ccategory'
            f'&page={page}'
            f'&layout_page_index={page}'
            '&text=смешарики'
        )

        api_url = (
            'https://www.ozon.ru/api/entrypoint-api.bx/page/json/v2'
            + '?'
            + urlencode({'url': search_url})
        )

        script = """
            const callback = arguments[arguments.length - 1];

            fetch(arguments[0], {
                method: 'GET',
                credentials: 'include'
            })
            .then(response => response.json())
            .then(data => callback(data))
            .catch(error => callback({'error': String(error)}));
        """

        data = driver.execute_async_script(script, api_url)

        if data.get('error'):
            print('Ошибка:', data.get('error'))
            break

        items = get_items_from_ozon_json(data)

        relevant_count = 0

        for item in items:
            product_id = item.get('sku') or item.get('id')

            if product_id in seen_ids:
                continue

            seen_ids.add(product_id)

            main_state = item.get('mainState', [])

            product_name = extract_name(main_state)
            brand = extract_brand(main_state)

            if not is_relevant_product(product_name, brand):
                continue

            price, discount_price = extract_prices(main_state)
            rating, reviews_count = extract_rating_reviews(main_state)

            link = item.get('action', {}).get('link')

            if link:
                link = 'https://www.ozon.ru' + link.split('?')[0]

            all_products.append({
                'product_id': product_id,
                'product_name': product_name,
                'brand': brand,
                'brand_id': None,
                'source_name': 'Поиск Смешарики + Оригинал',
                'source_type': 'original_search',
                'source_page': page,
                'seller': None,
                'seller_id': None,
                'seller_rating': None,
                'price': price,
                'discount_price': discount_price,
                'rating': rating,
                'reviews_count': reviews_count,
                'category_name': None,
                'subject_id': None,
                'subject_parent_id': None,
                'total_quantity': extract_total_quantity(main_state, item),
                'marketplace': 'Ozon',
                'link': link
            })

            relevant_count += 1

        print(
            f'page={page}: товаров в ответе {len(items)}, '
            f'релевантных новых {relevant_count}'
        )

        if len(items) == 0 or relevant_count == 0:
            empty_pages += 1
        else:
            empty_pages = 0

        if empty_pages >= 2:
            print('Две пустые страницы подряд. Останавливаемся.')
            break

        time.sleep(2)

    return all_products


## Блок 6. Настройка браузера и запуск сбора данных

На данном этапе выполняется настройка браузера Chrome для автоматизированной работы через Selenium.

После запуска браузера открывается страница поиска Ozon по запросу «Смешарики» с активированным фильтром «Оригинал». Далее запускается процедура автоматического сбора данных.

В процессе работы браузер используется для получения доступа к внутренним JSON-запросам маркетплейса и сбора информации о товарах.

In [65]:
options = webdriver.ChromeOptions()
options.add_argument('--start-maximized')
options.add_argument('--disable-blink-features=AutomationControlled')

options.add_experimental_option(
    'excludeSwitches',
    ['enable-automation']
)

options.add_experimental_option(
    'useAutomationExtension',
    False
)

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

driver.get(
    'https://www.ozon.ru/search/'
    '?brandcertified=t&from_global=true'
    '&opened=seller%2Ccategory'
    '&text=смешарики'
)

print('Открыли Ozon. Ждём загрузку страницы...')
time.sleep(10)

ozon_products = collect_ozon_original_search(
    driver=driver,
    max_pages=20
)

Открыли Ozon. Ждём загрузку страницы...
page=1: товаров в ответе 8, релевантных новых 8
page=2: товаров в ответе 8, релевантных новых 6
page=3: товаров в ответе 8, релевантных новых 7
page=4: товаров в ответе 8, релевантных новых 7
page=5: товаров в ответе 8, релевантных новых 8
page=6: товаров в ответе 8, релевантных новых 5
page=7: товаров в ответе 8, релевантных новых 5
page=8: товаров в ответе 8, релевантных новых 7
page=9: товаров в ответе 8, релевантных новых 4
page=10: товаров в ответе 8, релевантных новых 6
page=11: товаров в ответе 8, релевантных новых 6
page=12: товаров в ответе 8, релевантных новых 0
page=13: товаров в ответе 8, релевантных новых 1
page=14: товаров в ответе 8, релевантных новых 1
page=15: товаров в ответе 8, релевантных новых 1
page=16: товаров в ответе 8, релевантных новых 1
page=17: товаров в ответе 8, релевантных новых 0
page=18: товаров в ответе 8, релевантных новых 0
Две пустые страницы подряд. Останавливаемся.


## Блок 7. Формирование итоговой таблицы

После завершения сбора данных формируется единый DataFrame, содержащий сведения обо всех найденных товарах.

На данном этапе удаляются дубликаты товаров по идентификатору товара. Это необходимо для исключения повторяющихся записей и получения корректной итоговой выборки.

Дополнительно выполняется обработка цен. Если стандартная цена товара отсутствует, вместо неё используется цена со скидкой. Такой подход позволяет избежать появления пропусков и сохранить актуальную стоимость товара для последующего анализа.

In [66]:
df_ozon = pd.DataFrame(ozon_products)

if len(df_ozon) > 0:
    df_ozon = df_ozon.drop_duplicates(
        subset=['product_id']
    )

if len(df_ozon) > 0:
    df_ozon['price'] = (
        df_ozon['price']
        .fillna(df_ozon['discount_price'])
    )

## Блок 8. Дополнение данных о продавцах

Для каждого товара из сформированной выборки открывается отдельная карточка товара на сайте Ozon.

После загрузки страницы автоматически извлекается название магазина-продавца и сохраняется в таблицу. Данный этап позволяет получить дополнительную информацию, отсутствующую в исходной поисковой выдаче маркетплейса.

Полученные данные впоследствии используются для анализа продавцов и восстановления информации о брендах.

Это смая долгая часть - она занимает минимум 10 минут, так как посещает все страницы, у которых отсутсвует название бренда. 

In [67]:
print()
print('Заполняем продавцов со страниц товаров...')

for counter, (index, row) in enumerate(df_ozon.iterrows(), start=1):
    product_url = row['link']

    if pd.isna(product_url) or product_url == '':
        continue

    seller = get_seller_from_product_page(driver, product_url)

    df_ozon.loc[index, 'seller'] = seller

    print(
        f'{counter}/{len(df_ozon)}:',
        row['product_name'],
        '—',
        seller
    )

    time.sleep(2)


Заполняем продавцов со страниц товаров...
1/73: Смешарики. Официальная кулинарная книга | Корнилова Мария Викторовна, Зурабова Анастасия Михайловна — Торговый Дом "Эксмо...
2/73: Смешарики: цитаты, которые мы случайно выучили наизусть — Торговый Дом "Эксмо...
3/73: "Смешарики. Лучшие моменты"/Настольная игра/Развивающая игра на счёт и общение для детей от 5 лет/Стиль Жизни — «СТИЛЬ ЖИЗНИ» - офи...
4/73: "Смешарики. Бесконечный праздник"/Настольная игра/Развивающая компактная игра на память и внимание для детей от 5 лет/Стиль Жизни — «СТИЛЬ ЖИЗНИ» - офи...
5/73: Игрушка-подвеска "Мякиши" Крош Смешарики, игрушки для детей — МЯКИШИ фабрика игру...
6/73: Смешарики сквозь вселенные. Пролог | Яров Николай Сергеевич — Торговый Дом "Эксмо...
7/73: Игрушка-подвеска "Мякиши" Ёжик, Совунья, Карыч, Смешарики, игрушки для детей 3 штуки — МЯКИШИ фабрика игру...
8/73: Мягкие игрушки "Мякиши" Крошик Малышарики, детские игрушки, 0+ — МЯКИШИ фабрика игру...
9/73: Смешарики. История культовой Вселенной 

## Блок 9. Восстановление отсутствующих брендов

В процессе анализа было обнаружено, что часть товаров не содержит информации о бренде в поисковой выдаче Ozon.

Для повышения качества итогового датасета был сформирован словарь соответствия между официальными магазинами и брендами. Если бренд товара отсутствует, но известно название продавца, бренд автоматически восстанавливается на основании названия магазина.

Такой подход позволяет существенно сократить количество пропусков в данных и повысить точность последующего анализа структуры рынка.

In [68]:
brand_mapping = {
    'Торговый Дом "Эксмо': 'Эксмо',
    'МЯКИШИ': 'Мякиши',
    'Befree': 'Befree',
    'ХАТБЕР': 'Hatber',
    'Играмир': 'Играмир',
    'СТИЛЬ ЖИЗНИ': 'Стиль Жизни',
    'Scandic': 'Scandic'
}

for seller_pattern, brand_name in brand_mapping.items():

    mask = (
        df_ozon['brand'].isna()
        &
        df_ozon['seller'].fillna('').str.contains(
            seller_pattern,
            case=False,
            regex=False
        )
    )

    df_ozon.loc[mask, 'brand'] = brand_name

## Блок 10. Сохранение результатов и проверка качества данных

После завершения обработки итоговый датасет сохраняется в форматах CSV и XLSX для дальнейшего анализа.

Для контроля качества выполняется первичная проверка данных. Выводятся первые строки таблицы, что позволяет визуально убедиться в корректности извлечения основных характеристик товаров, брендов, продавцов и цен.

Полученный датасет используется на следующих этапах исследования для анализа ассортимента, ценовой политики и структуры рынка лицензионной продукции бренда «Смешарики» на маркетплейсе Ozon.

In [70]:
df_ozon.to_csv(
    'smeshariki_ozon_final.csv',
    index=False,
    encoding='utf-8-sig'
)

df_ozon.to_excel(
    'smeshariki_ozon_final.xlsx',
    index=False
)

print('Первые товары:')

display(
    df_ozon[
        [
            'product_name',
            'brand',
            'seller',
            'price',
            'discount_price',
            'rating',
            'reviews_count',
            'link'
        ]
    ].head(10)
)

Первые товары:


,product_name,brand,seller,price,discount_price,rating,reviews_count,link
0,Смешарики. Официальная кулинарная книга | Корн...,Эксмо,"Торговый Дом ""Эксмо...",1157.0,1157.0,5.0,279.0,https://www.ozon.ru/product/smeshariki-ofitsia...
1,"Смешарики: цитаты, которые мы случайно выучили...",Эксмо,"Торговый Дом ""Эксмо...",895.0,895.0,4.9,396.0,https://www.ozon.ru/product/smeshariki-tsitaty...
2,"""Смешарики. Лучшие моменты""/Настольная игра/Ра...",Стиль Жизни,«СТИЛЬ ЖИЗНИ» - офи...,554.0,554.0,5.0,1659.0,https://www.ozon.ru/product/smeshariki-luchshi...
3,"""Смешарики. Бесконечный праздник""/Настольная и...",Стиль Жизни,«СТИЛЬ ЖИЗНИ» - офи...,632.0,632.0,5.0,1659.0,https://www.ozon.ru/product/smeshariki-beskone...
4,"Игрушка-подвеска ""Мякиши"" Крош Смешарики, игру...",Мякиши,МЯКИШИ фабрика игру...,1550.0,290.0,4.9,5435.0,https://www.ozon.ru/product/igrushka-podveska-...
5,Смешарики сквозь вселенные. Пролог | Яров Нико...,Эксмо,"Торговый Дом ""Эксмо...",231.0,231.0,NaN,NaN,https://www.ozon.ru/product/smeshariki-skvoz-v...
6,"Игрушка-подвеска ""Мякиши"" Ёжик, Совунья, Карыч...",Мякиши,МЯКИШИ фабрика игру...,3850.0,795.0,4.9,5435.0,https://www.ozon.ru/product/igrushka-podveska-...
7,"Мягкие игрушки ""Мякиши"" Крошик Малышарики, дет...",Мякиши,МЯКИШИ фабрика игру...,1064.0,1064.0,5.0,3286.0,https://www.ozon.ru/product/myagkie-igrushki-m...
8,Смешарики. История культовой Вселенной | Корни...,Эксмо,"Торговый Дом ""Эксмо...",1843.0,1843.0,5.0,2075.0,https://www.ozon.ru/product/smeshariki-istoriy...
9,"Игрушка-подвеска ""Мякиши"" Ёжик Смешарики",Мякиши,МЯКИШИ фабрика игру...,368.0,368.0,4.9,5435.0,https://www.ozon.ru/product/igrushka-podveska-...


## 4. Объединение данных Wildberries и Ozon

In [71]:
# Объединение данных маркетплейсов
print('Размер WB:', df_wb.shape)
print('Размер Ozon:', df_ozon.shape)

df_marketplaces = pd.concat(
    [df_wb, df_ozon],
    ignore_index=True
)

df_marketplaces = df_marketplaces.drop_duplicates(
    subset=['marketplace', 'product_id']
).copy()

print('Размер общего датасета:')
print(df_marketplaces.shape)

display(df_marketplaces.head())

# проверка распределения
print('Товары по маркетплейсам:')

display(
    df_marketplaces['marketplace']
    .value_counts()
)

# Сохранение общего файла
df_marketplaces.to_csv(
    'smeshariki_marketplaces_final.csv',
    index=False,
    encoding='utf-8-sig'
)

df_marketplaces.to_excel(
    'smeshariki_marketplaces_final.xlsx',
    index=False
)

print(
    'Файлы сохранены: '
    'smeshariki_marketplaces_final.csv и '
    'smeshariki_marketplaces_final.xlsx'
)

Размер WB: (2835, 20)
Размер Ozon: (73, 20)
Размер общего датасета:
(2908, 20)


,product_id,product_name,brand,brand_id,source_name,source_type,source_page,seller,seller_id,seller_rating,price,discount_price,rating,reviews_count,category_name,subject_id,subject_parent_id,total_quantity,marketplace,link
0,183944199,Мультивселенная. Комиксы BUBBLE,Смешарики,1187,Смешарики,brand_page,NaN,BUBBLE,38137,4.9,2608.0,990.0,4.9,792.0,,4961,786,40.0,Wildberries,https://www.wildberries.ru/catalog/183944199/d...
1,158386267,Школьные мелки 10 цветов 29 штук,Смешарики,1187,Смешарики,brand_page,NaN,ТойсМаркет,339332,4.9,535.0,168.0,4.3,241.0,,714,571,40.0,Wildberries,https://www.wildberries.ru/catalog/158386267/d...
2,217050192,Маленькая мягкая игрушка брелок Крош на рюкзак,Смешарики,1187,Смешарики,brand_page,NaN,Фабрика игрушек МЯКИШИ.,27411,4.9,1145.0,385.0,4.9,1155.0,игрушки-подвески,268,7,40.0,Wildberries,https://www.wildberries.ru/catalog/217050192/d...
3,256270492,Термостакан 350 мл с клапаном,Смешарики,1187,Смешарики,brand_page,NaN,Бытпласт - товары для дома из пластика,80145,4.9,1200.0,331.0,4.9,350.0,,1274,1590,40.0,Wildberries,https://www.wildberries.ru/catalog/256270492/d...
4,217080342,Маленькая мягкая игрушка Пин на рюкзак,Смешарики,1187,Смешарики,brand_page,NaN,Фабрика игрушек МЯКИШИ.,27411,4.9,1436.0,398.0,4.9,696.0,мягкие игрушки,268,7,40.0,Wildberries,https://www.wildberries.ru/catalog/217080342/d...


Товары по маркетплейсам:


marketplace
Wildberries    2835
Ozon             73
Name: count, dtype: int64

Файлы сохранены: smeshariki_marketplaces_final.csv и smeshariki_marketplaces_final.xlsx
